# STAGE 1: Data Preparation & Ground Truth Generation
**Dataset:** Statlog (German Credit Data) — UCI ML Repository (ID: 144)  
**Goal:** Load data, perform descriptive analysis, engineer features, generate Ground Truth labels, and produce clean train/test splits ready for Random Forest training.




## 0. Install dependencies

In [87]:
# Install the UCI ML Repo helper if not already present
!pip install ucimlrepo --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Reproducibility seed used throughout the notebook
RANDOM_STATE = 42

# ── Colour palette ────────────────────────────────────────────────────────────
# Each colour has a fixed meaning used consistently in every chart:
#   C_DENY    = '#EF4323'  denial, bad outcome, high cost / worse model
#   C_APPROVE = '#5804FA'  approval, good outcome / better model
#   C_BEST    = '#22E563'  best value in a comparison
#   C_INK     = '#131313'  axis text and lines
#   C_BG      = '#F9F9F9'  figure background
C_DENY    = '#EF4323'
C_APPROVE = '#5804FA'
C_BEST    = '#22E563'
C_INK     = '#131313'
C_BG      = '#F9F9F9'

# Apply global style defaults
plt.rcParams.update({
    'figure.dpi'          : 120,
    'figure.facecolor'    : C_BG,
    'axes.facecolor'      : C_BG,
    'axes.edgecolor'      : C_INK,
    'axes.labelcolor'     : C_INK,
    'xtick.color'         : C_INK,
    'ytick.color'         : C_INK,
    'text.color'          : C_INK,
    'axes.grid'           : False,
    'font.family'         : 'sans-serif',
})

print('All imports OK')


---
## 1. Download Data
We fetch the dataset directly from the UCI ML Repository using `ucimlrepo`.
- **X** — 20 feature columns (Attribute1 … Attribute20)
- **y** — target column `class` (1 = Good credit risk, 2 = Bad credit risk)

In [89]:
# ── Fetch dataset ──────────────────────────────────────────────────────────────
dataset = fetch_ucirepo(id=144)

X_raw = dataset.data.features.copy()   # 1000 × 20
y_raw = dataset.data.targets.copy()    # 1000 × 1  (column name: 'class')

# Flatten target to a 1-D Series and rename for clarity
y_raw = y_raw['class'].rename('class')

# Combine into one working DataFrame for EDA
df = X_raw.copy()
df['class'] = y_raw.values

print(f'Dataset shape: {df.shape}')
print(f"Class distribution (1=Good, 2=Bad):\n{df['class'].value_counts()}")
df.head()

Dataset shape: (1000, 21)
Class distribution (1=Good, 2=Bad):
class
1    700
2    300
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [90]:
# Quick schema overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Attribute1   1000 non-null   object
 1   Attribute2   1000 non-null   int64 
 2   Attribute3   1000 non-null   object
 3   Attribute4   1000 non-null   object
 4   Attribute5   1000 non-null   int64 
 5   Attribute6   1000 non-null   object
 6   Attribute7   1000 non-null   object
 7   Attribute8   1000 non-null   int64 
 8   Attribute9   1000 non-null   object
 9   Attribute10  1000 non-null   object
 10  Attribute11  1000 non-null   int64 
 11  Attribute12  1000 non-null   object
 12  Attribute13  1000 non-null   int64 
 13  Attribute14  1000 non-null   object
 14  Attribute15  1000 non-null   object
 15  Attribute16  1000 non-null   int64 
 16  Attribute17  1000 non-null   object
 17  Attribute18  1000 non-null   int64 
 18  Attribute19  1000 non-null   object
 19  Attribute20  1000 non-null  

---
## 2. Descriptive Analysis (pre-intervention)

Before any feature engineering we visualise approval/denial patterns across
demographic axes: **gender**, **age**, and **marital status** (derived from Attribute9).

### Attribute9 codes
| Code | Meaning |
|------|---------|
| A91  | male : divorced/separated |
| A92  | female : divorced/separated/married |
| A93  | male : single |
| A94  | male : married/widowed |
| A95  | female : single |

In [91]:
# ── Helper: map Attribute9 codes to readable labels ────────────────────────────
A9_SEX = {
    'A91': 'male',
    'A92': 'female',
    'A93': 'male',
    'A94': 'male',
    'A95': 'female',
}
A9_MARITAL = {
    'A91': 'divorced/separated',
    'A92': 'divorced/separated/married',
    'A93': 'single',
    'A94': 'married/widowed',
    'A95': 'single',
}
A9_LABEL = {
    'A91': 'M: div/sep',
    'A92': 'F: div/sep/mar',
    'A93': 'M: single',
    'A94': 'M: mar/wid',
    'A95': 'F: single',
}

# Add temporary readable columns for EDA (not yet the engineered features)
df['_sex']     = df['Attribute9'].map(A9_SEX)
df['_marital'] = df['Attribute9'].map(A9_MARITAL)
df['_a9_label']= df['Attribute9'].map(A9_LABEL)
df['_age']     = df['Attribute13'].astype(int)   # Attribute13 = age in years
df['_is_young']= (df['_age'] <= 25).map({True: 'Age ≤ 25', False: 'Age > 25'})
df['_approved']= (df['class'] == 1).map({True: 'Approved', False: 'Denied'})

print('Temporary EDA columns added.')

Temporary EDA columns added.


In [ ]:
# Human-readable labels for dataset attributes.
# These are used only in charts — the dataframe keeps the original column names.
ATTR_LABEL = {
    'Attribute1'  : 'Checking Account Status',
    'Attribute2'  : 'Loan Duration (months)',
    'Attribute3'  : 'Credit History',
    'Attribute4'  : 'Loan Purpose',
    'Attribute5'  : 'Loan Amount',
    'Attribute6'  : 'Savings Account',
    'Attribute7'  : 'Employment Duration',
    'Attribute8'  : 'Installment Rate (% income)',
    'Attribute9'  : 'Personal Status & Sex',
    'Attribute10' : 'Other Debtors / Guarantors',
    'Attribute11' : 'Residence Duration (years)',
    'Attribute12' : 'Property',
    'Attribute13' : 'Age (years)',
    'Attribute14' : 'Other Installment Plans',
    'Attribute15' : 'Housing',
    'Attribute16' : 'Number of Existing Credits',
    'Attribute17' : 'Job Type',
    'Attribute18' : 'Number of Dependents',
    'Attribute19' : 'Telephone Registered',
    'Attribute20' : 'Foreign Worker',
    # Engineered features
    'is_young'    : 'Young Applicant (Age <= 25)',
    'sex_male'    : 'Sex: Male',
    'sex_female'  : 'Sex: Female',
    'marital_status_single'                    : 'Marital: Single',
    'marital_status_married/widowed'           : 'Marital: Married/Widowed',
    'marital_status_divorced/separated'        : 'Marital: Divorced/Separated',
    'marital_status_divorced/separated/married': 'Marital: Div/Sep/Married',
}

def readable(col):
    """Return a human-readable label for a feature column name."""
    for key, label in ATTR_LABEL.items():
        if col == key or col.startswith(key + '_'):
            return label
    return col


In [ ]:
# Approval and denial counts by gender.
# C_APPROVE = approved (blue-violet), C_DENY = denied (red-orange).
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=C_BG)

sex_counts = df.groupby(['_sex', '_approved']).size().unstack(fill_value=0)
sex_counts.plot(kind='bar', ax=axes[0],
                color=[C_APPROVE, C_DENY], edgecolor='white')
axes[0].set_title('Approval / Denial by Gender (counts)', fontweight='bold', color=C_INK)
axes[0].set_xlabel('Gender', color=C_INK)
axes[0].set_ylabel('Number of applicants', color=C_INK)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='')
axes[0].grid(False)

sex_pct = sex_counts.div(sex_counts.sum(axis=1), axis=0) * 100
sex_pct.plot(kind='bar', stacked=True, ax=axes[1],
             color=[C_APPROVE, C_DENY], edgecolor='white')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Approval / Denial by Gender (%)', fontweight='bold', color=C_INK)
axes[1].set_xlabel('Gender', color=C_INK)
axes[1].set_ylabel('Share of applicants', color=C_INK)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='', loc='lower right')
axes[1].grid(False)

plt.tight_layout()
plt.savefig('plot_gender.png', bbox_inches='tight')
plt.show()

print(sex_pct.round(1).to_string())


In [ ]:
# Approval and denial by age bracket.
bins   = [0, 25, 35, 45, 55, 120]
labels = ['<=25', '26-35', '36-45', '46-55', '55+']
df['_age_group'] = pd.cut(df['_age'], bins=bins, labels=labels, right=True)

age_counts = df.groupby(['_age_group', '_approved'], observed=True).size().unstack(fill_value=0)
age_pct    = age_counts.div(age_counts.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=C_BG)

age_counts.plot(kind='bar', ax=axes[0],
                color=[C_APPROVE, C_DENY], edgecolor='white')
axes[0].set_title('Approval / Denial by Age Group (counts)', fontweight='bold', color=C_INK)
axes[0].set_xlabel('Age Group', color=C_INK)
axes[0].set_ylabel('Number of applicants', color=C_INK)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='')
axes[0].grid(False)

age_pct.plot(kind='bar', stacked=True, ax=axes[1],
             color=[C_APPROVE, C_DENY], edgecolor='white')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Approval / Denial by Age Group (%)', fontweight='bold', color=C_INK)
axes[1].set_xlabel('Age Group', color=C_INK)
axes[1].set_ylabel('Share of applicants', color=C_INK)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='', loc='lower right')
axes[1].grid(False)

plt.tight_layout()
plt.savefig('plot_age_group.png', bbox_inches='tight')
plt.show()


In [ ]:
# Approval and denial by marital status.
mar_counts = df.groupby(['_marital', '_approved']).size().unstack(fill_value=0)
mar_pct    = mar_counts.div(mar_counts.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=C_BG)

mar_counts.plot(kind='bar', ax=axes[0],
                color=[C_APPROVE, C_DENY], edgecolor='white')
axes[0].set_title('Approval / Denial by Marital Status (counts)', fontweight='bold', color=C_INK)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of applicants', color=C_INK)
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='')
axes[0].grid(False)

mar_pct.plot(kind='bar', stacked=True, ax=axes[1],
             color=[C_APPROVE, C_DENY], edgecolor='white')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Approval / Denial by Marital Status (%)', fontweight='bold', color=C_INK)
axes[1].set_xlabel('')
axes[1].set_ylabel('Share of applicants', color=C_INK)
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='', loc='lower right')
axes[1].grid(False)

plt.tight_layout()
plt.savefig('plot_marital.png', bbox_inches='tight')
plt.show()


In [ ]:
# Summary of approval rate gaps across all three demographic attributes.
# P-values from chi-squared tests on approval/denial contingency tables.

def chi2_pvalue(df_src, group_col, outcome_col, approved_label='Approved'):
    """Return the chi-squared p-value for independence of group_col and outcome_col."""
    ct = pd.crosstab(df_src[group_col], df_src[outcome_col] == approved_label)
    _, p, _, _ = stats.chi2_contingency(ct, correction=False)
    return p

gap_rows = []

sex_approved = df.groupby('_sex')['_approved'].apply(
    lambda s: (s == 'Approved').mean() * 100)
gap_rows.append({
    'Attribute'     : 'Sex (Male vs Female)',
    'Group A'       : 'Male',
    'Approval A (%)': round(sex_approved.get('male',   0), 1),
    'Group B'       : 'Female',
    'Approval B (%)': round(sex_approved.get('female', 0), 1),
    'Gap (pp)'      : round(abs(sex_approved.get('male', 0) -
                                sex_approved.get('female', 0)), 1),
    'p-value'       : chi2_pvalue(df, '_sex', '_approved'),
})

young_approved = df.groupby('_is_young')['_approved'].apply(
    lambda s: (s == 'Approved').mean() * 100)
gap_rows.append({
    'Attribute'     : 'Age (Young ≤25 vs Other)',
    'Group A'       : 'Age > 25',
    'Approval A (%)': round(young_approved.get('Age > 25', 0), 1),
    'Group B'       : 'Age ≤ 25',
    'Approval B (%)': round(young_approved.get('Age ≤ 25', 0), 1),
    'Gap (pp)'      : round(abs(young_approved.get('Age > 25', 0) -
                                young_approved.get('Age ≤ 25', 0)), 1),
    'p-value'       : chi2_pvalue(df, '_is_young', '_approved'),
})

mar_approved = df.groupby('_marital')['_approved'].apply(
    lambda s: (s == 'Approved').mean() * 100)
gap_rows.append({
    'Attribute'     : 'Marital Status (max spread)',
    'Group A'       : mar_approved.idxmax(),
    'Approval A (%)': round(mar_approved.max(), 1),
    'Group B'       : mar_approved.idxmin(),
    'Approval B (%)': round(mar_approved.min(), 1),
    'Gap (pp)'      : round(mar_approved.max() - mar_approved.min(), 1),
    'p-value'       : chi2_pvalue(df, '_marital', '_approved'),
})

gap_df = pd.DataFrame(gap_rows).set_index('Attribute')

print("=" * 78)
print("  EDA SUMMARY — Approval Rate Gaps by Demographic Attribute")
print("=" * 78)
display(gap_df)
print()
print("Interpretation (significance threshold α = 0.05):")
for attr, row in gap_df.iterrows():
    sig = "  *" if row['p-value'] < 0.05 else "   (not significant)"
    print(f"  {attr:<38} gap = {row['Gap (pp)']:>5.1f} pp   p = {row['p-value']:.4f}{sig}")
print()
print("Note: only attributes marked * have a statistically significant difference")
print("in approval rates. The fairness focus for Stages 2-5 follows from both")
print("the magnitude of the gap and its statistical significance.")

In [ ]:
# Age group approval rates — the primary fairness focus of this project.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=C_BG)

young_counts = df.groupby(['_is_young', '_approved']).size().unstack(fill_value=0)
young_pct    = young_counts.div(young_counts.sum(axis=1), axis=0) * 100

young_counts.plot(kind='bar', ax=axes[0],
                  color=[C_APPROVE, C_DENY], edgecolor='white')
axes[0].set_title('Approval / Denial: Young vs Others (counts)',
                  fontweight='bold', color=C_INK)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of applicants', color=C_INK)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='')
axes[0].grid(False)

young_pct.plot(kind='bar', stacked=True, ax=axes[1],
               color=[C_APPROVE, C_DENY], edgecolor='white')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Approval / Denial: Young vs Others (%)',
                  fontweight='bold', color=C_INK)
axes[1].set_xlabel('')
axes[1].set_ylabel('Share of applicants', color=C_INK)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='', loc='lower right')
axes[1].grid(False)

plt.tight_layout()
plt.savefig('plot_young.png', bbox_inches='tight')
plt.show()

print(young_pct.round(1).to_string())
approval_gap = young_pct.loc['Age > 25', 'Approved'] - young_pct.loc['Age ≤ 25', 'Approved']
print(f"\nApproval gap: {young_pct.loc['Age > 25','Approved']:.1f}% vs "
      f"{young_pct.loc['Age ≤ 25','Approved']:.1f}% = {approval_gap:.1f} pp")


---
## 3. Feature Engineering

Steps:
1. **Split Attribute9** into `sex` and `marital_status`.
2. **Create `is_young`** binary flag (Age ≤ 25).
3. **One-hot encode** all categorical columns with `pd.get_dummies`.
4. Drop the temporary EDA columns.

In [97]:
# ── 3.1 Start from the raw feature matrix ──────────────────────────────────────
feat = X_raw.copy()

# ── 3.2 Decompose Attribute9 → sex + marital_status ────────────────────────────
feat['sex'] = feat['Attribute9'].map(A9_SEX)
feat['marital_status'] = feat['Attribute9'].map(A9_MARITAL)

# Drop the original combined column — it is now fully captured by sex + marital_status
feat.drop(columns=['Attribute9'], inplace=True)

print('sex distribution:')
print(feat['sex'].value_counts())
print('\nmarital_status distribution:')
print(feat['marital_status'].value_counts())

sex distribution:
sex
male      690
female    310
Name: count, dtype: int64

marital_status distribution:
marital_status
single                        548
divorced/separated/married    310
married/widowed                92
divorced/separated             50
Name: count, dtype: int64


In [98]:
# ── 3.3 Binary flag: is_young (Age ≤ 25) ───────────────────────────────────────
# Attribute13 = age in years
feat['is_young'] = (feat['Attribute13'].astype(int) <= 25).astype(int)

print(f"Young applicants (Age ≤ 25): {feat['is_young'].sum()} / {len(feat)}")

Young applicants (Age ≤ 25): 190 / 1000


In [99]:
# ── 3.4 One-hot encode all object/categorical columns ──────────────────────────
# Identify categorical columns that still remain as strings
cat_cols = feat.select_dtypes(include=['object', 'category']).columns.tolist()
print('Columns to encode:', cat_cols)

feat_encoded = pd.get_dummies(feat, columns=cat_cols, drop_first=False, dtype=int)

print(f'\nShape after encoding: {feat_encoded.shape}')
feat_encoded.head()

Columns to encode: ['Attribute1', 'Attribute3', 'Attribute4', 'Attribute6', 'Attribute7', 'Attribute10', 'Attribute12', 'Attribute14', 'Attribute15', 'Attribute17', 'Attribute19', 'Attribute20', 'sex', 'marital_status']

Shape after encoding: (1000, 64)


,Attribute2,Attribute5,Attribute8,Attribute11,Attribute13,Attribute16,Attribute18,is_young,Attribute1_A11,Attribute1_A12,...,Attribute19_A191,Attribute19_A192,Attribute20_A201,Attribute20_A202,sex_female,sex_male,marital_status_divorced/separated,marital_status_divorced/separated/married,marital_status_married/widowed,marital_status_single
0,6,1169,4,4,67,2,1,0,1,0,...,0,1,1,0,0,1,0,0,0,1
1,48,5951,2,2,22,1,1,1,0,1,...,1,0,1,0,1,0,0,1,0,0
2,12,2096,2,3,49,1,2,0,0,0,...,1,0,1,0,0,1,0,0,0,1
3,42,7882,2,4,45,1,2,0,1,0,...,1,0,1,0,0,1,0,0,0,1
4,24,4870,3,4,53,2,2,0,1,0,...,1,0,1,0,0,1,0,0,0,1


---
## 4. Target Variable

The UCI dataset already contains the bank's credit decision for all 1,000 applicants:

- **class = 1** → bank rated the applicant as **Good** (creditworthy)
- **class = 2** → bank rated the applicant as **Bad** (not creditworthy)

We use this directly as our target variable. The model learns to reproduce the
bank's classification logic — and then we audit whether that logic treats
demographic groups equally.

`y = 1` means **Bad / flagged** (the bank denied or would deny the application).
`y = 0` means **Good / approved** (the bank considered the applicant creditworthy).

No simulation is needed. The bank has already labelled all 1,000 cases.


In [ ]:
# ── 4. Define target variable from bank decisions ──────────────────────────────
# class == 2  →  Bad applicant  →  y = 1  (model should flag / deny)
# class == 1  →  Good applicant →  y = 0  (model should approve)

gt_df = feat_encoded.copy()
gt_df['class'] = y_raw.values
gt_df['y']     = (gt_df['class'] == 2).astype(int)   # 1 = bank said Bad

print(f"Total applicants      : {len(gt_df)}")
print(f"Bank-rated Bad  (y=1) : {gt_df['y'].sum()}  ({gt_df['y'].mean():.1%})")
print(f"Bank-rated Good (y=0) : {(gt_df['y'] == 0).sum()}  ({(gt_df['y'] == 0).mean():.1%})")


---
## 5. Train / Test Split

We split all 1,000 applicants into 80% training and 20% test sets,
stratified on `y` so that both splits contain the same proportion of
Good and Bad applicants.


In [ ]:
# ── 5. Stratified 80/20 split on all 1,000 applicants ─────────────────────────
feature_cols = [c for c in feat_encoded.columns]

X_all  = gt_df[feature_cols]
y_gt   = gt_df['y']                  # target: 1 = bank said Bad
y_bank = gt_df['class']              # raw bank label (1=Good, 2=Bad)

X_train, X_test, y_train, y_gt_test, y_bank_train, y_bank_test = train_test_split(
    X_all, y_gt, y_bank,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_gt,
)

print('─' * 50)
print(f'X_train shape      : {X_train.shape}')
print(f'X_test  shape      : {X_test.shape}')
print(f'y_train Bad %      : {y_train.mean():.1%}')
print(f'y_test  Bad %      : {y_gt_test.mean():.1%}   (stratified)')
print('─' * 50)
print('All global variables ready for Random Forest training.')


In [ ]:
# ── Final verification ─────────────────────────────────────────────────────────
assert X_train.isnull().sum().sum() == 0, 'NaNs in X_train!'
assert X_test.isnull().sum().sum()  == 0, 'NaNs in X_test!'
assert y_train.isnull().sum()       == 0, 'NaNs in y_train!'
assert y_gt_test.isnull().sum()     == 0, 'NaNs in y_gt_test!'

non_numeric = X_train.select_dtypes(exclude=['number']).columns.tolist()
assert len(non_numeric) == 0, f'Non-numeric columns remain: {non_numeric}'

print(' All assertions passed.')
print(f'   Features   : {X_train.shape[1]}')
print(f'   Train rows : {len(X_train)}')
print(f'   Test  rows : {len(X_test)}')


---
# STAGE 2: Baseline Model — Random Forest (No Fairness Intervention)

We train a Random Forest to replicate the bank's credit decisions,
then evaluate it on two dimensions:

1. **Predictive performance** — how accurately does the model reproduce the
   bank's Good/Bad labels? (ROC-AUC, Accuracy)
2. **Fairness** — does the model distribute Bad labels equally across
   demographic groups, and how does this compare to the bank itself?

The bank's own decisions serve as the reference point for fairness, not for
financial performance — since the bank is by definition "correct" on its
own labels.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

def dp_gap(y_pred, sensitive_col):
    """
    Compute the Demographic Parity gap for a binary sensitive attribute.
    Returns a DataFrame with flag rates per group and the absolute gap.
    """
    groups = sorted(sensitive_col.unique())
    rows   = []
    for g in groups:
        mask      = (sensitive_col == g)
        flag_rate = pd.Series(y_pred, index=sensitive_col.index)[mask].mean()
        rows.append({'group': g, 'n': int(mask.sum()), 'flag_rate': flag_rate})
    df_g = pd.DataFrame(rows).set_index('group')
    df_g['DP_gap'] = abs(df_g['flag_rate'].iloc[0] - df_g['flag_rate'].iloc[1])
    return df_g

print("Stage 2 imports and dp_gap() ready.")


## 2.1 — Train the Baseline Model

We train a Random Forest to replicate the bank's credit decisions —
predicting whether the bank would classify an applicant as **Bad (y=1)** or
**Good (y=0)**.

The hyperparameters below were chosen to reduce overfitting (see max_depth,
min_samples_leaf) while keeping the 1:5 cost asymmetry via class_weight.


In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=7,
    max_features='log2',
    criterion='entropy',
    class_weight='balanced_subsample',   # up-weights minority class (Bad)
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_baseline.fit(X_train, y_train)

print("Baseline RF trained.")
print(f"  Trees         : {rf_baseline.n_estimators}")
print(f"  Max depth     : {rf_baseline.max_depth}")
print(f"  Features used : {X_train.shape[1]}")
print(f"  Train samples : {len(X_train)}")


## 2.2 — Predictions

We apply a decision threshold of **0.4**: if the model's estimated probability
that an applicant is Bad exceeds 40%, we flag them.

This is slightly above the default 0.5, reflecting that the model already
tends toward caution via `class_weight`. Adjust `THRESHOLD` to explore
the performance-vs-fairness tradeoff.


In [ ]:
THRESHOLD = 0.4

y_prob_baseline = rf_baseline.predict_proba(X_test)[:, 1]         # P(Bad=1)
y_pred_baseline = (y_prob_baseline >= THRESHOLD).astype(int)       # 1 = flagged as Bad

print(f"Decision threshold : {THRESHOLD}")
print(f"Flagged as Bad     : {y_pred_baseline.sum()} / {len(y_pred_baseline)}")


## 2.3 — Performance Metrics

We compare the model's flags against the bank's actual decisions (`y_gt_test`).

- **True Positive**: model correctly flags an applicant the bank also rated Bad
- **False Positive**: model flags an applicant the bank rated Good (over-caution)
- **False Negative**: model approves an applicant the bank rated Bad (under-caution)
- **True Negative**: model correctly approves a Good applicant

The ROC-AUC shows how well the model separates the two classes overall.


In [ ]:
from sklearn.metrics import classification_report

auc_train    = roc_auc_score(y_train, rf_baseline.predict_proba(X_train)[:, 1])
auc_test     = roc_auc_score(y_gt_test, y_prob_baseline)
acc_baseline = accuracy_score(y_gt_test, y_pred_baseline)
cm_baseline  = confusion_matrix(y_gt_test, y_pred_baseline)

tn, fp, fn, tp = cm_baseline.ravel()

print("=" * 52)
print("  BASELINE MODEL — PERFORMANCE METRICS")
print("=" * 52)
print(f"  ROC-AUC  (train) : {auc_train:.4f}")
print(f"  ROC-AUC  (test)  : {auc_test:.4f}")
print(f"  Accuracy (test)  : {acc_baseline:.4f}")
print("-" * 52)
print(f"  Confusion Matrix  (threshold = {THRESHOLD})")
print(f"  {'':20s}  Pred: Good  Pred: Bad")
print(f"  Actual: Good            {tn:>5}      {fp:>5}")
print(f"  Actual: Bad             {fn:>5}      {tp:>5}")
print("-" * 52)
print(classification_report(y_gt_test, y_pred_baseline,
                             target_names=['Good (0)', 'Bad (1)']))


## 2.3 — Performance: How Well Does the Model Reproduce Bank Decisions?

**ROC-AUC** measures the model's ability to rank Bad applicants above Good ones.
An AUC of 1.0 would mean perfect reproduction of bank decisions; 0.5 is random.

**Accuracy** is the share of applicants labelled identically by both the model
and the bank. Note that accuracy alone can be misleading given the class
imbalance (70% Good, 30% Bad).


In [ ]:
# Performance comparison: model vs bank decisions.
# The bank's AUC = 1.0 and Accuracy = 1.0 by definition — it is the ground truth.
# We report these only to show the gap the model has to close.

bank_pred_binary = (y_bank_test == 2).astype(int).values

auc_bank  = roc_auc_score(y_gt_test, bank_pred_binary)
acc_bank  = accuracy_score(y_gt_test, bank_pred_binary)

print("=" * 54)
print("  PERFORMANCE SUMMARY")
print("=" * 54)
print(f"  {'Metric':<28} {'Bank':>8}  {'Baseline RF':>11}")
print("-" * 54)
print(f"  {'ROC-AUC':<28} {auc_bank:>8.4f}  {auc_test:>11.4f}")
print(f"  {'Accuracy':<28} {acc_bank:>8.4f}  {acc_baseline:>11.4f}")
print("=" * 54)
print()
print("  Note: Bank AUC = 1.0 because the bank's labels ARE the target.")
print("  The model learns to approximate these decisions, not surpass them.")
print(f"  A test AUC of {auc_test:.3f} means the model has learned the bank's")
print("  decision logic reasonably well, but not perfectly.")


## 2.4 — Fairness Audit: Demographic Parity (Age)

We measure the **Demographic Parity gap** for age — consistent with the
decision made in Stage 1 to focus on age as the primary protected attribute.

```
DP gap (age) = |Bad-label rate for Age ≤ 25 − Bad-label rate for Age > 25|
```

- **Bank DP gap** — how unequally the bank historically labelled young vs. other applicants
- **Model DP gap** — whether the model inherits, reduces, or amplifies that inequality


In [ ]:
young_test       = X_test['is_young']
bank_pred_binary = (y_bank_test == 2).astype(int).values

dp_age_bank = dp_gap(bank_pred_binary, young_test)
dp_age_base = dp_gap(y_pred_baseline,  young_test)

gap_bank = dp_age_bank['DP_gap'].iloc[0]
gap_base = dp_age_base['DP_gap'].iloc[0]

print("=" * 56)
print("  DEMOGRAPHIC PARITY AUDIT — AGE")
print("=" * 56)
for label, df_g in [('Bank decisions', dp_age_bank),
                    ('Baseline RF',    dp_age_base)]:
    print(f"\n  {label}")
    print(f"  {'Group':<12} {'n':>5}  {'Bad-label rate':>14}")
    print("  " + "-" * 34)
    for g in df_g.index:
        grp_label = 'Age <= 25' if g == 1 else 'Age > 25'
        print(f"  {grp_label:<12} {int(df_g.loc[g,'n']):>5}  {df_g.loc[g,'flag_rate']:>14.3f}")
    print(f"  DP gap: {df_g['DP_gap'].iloc[0]:.3f}")

print(f"\n  Bank DP gap (age) : {gap_bank:.3f}")
print(f"  Model DP gap (age): {gap_base:.3f}  "
      f"({'amplified' if gap_base > gap_bank else 'reduced'} vs bank)")


## 2.5 — Summary Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=C_BG)

# --- Panel A: Prediction outcomes --------------------------------------------
ax = axes[0]
tn, fp, fn, tp = cm_baseline.ravel()
outcomes = ['True Neg\n(correct Good)', 'False Pos\n(over-flagged)',
            'False Neg\n(missed Bad)',  'True Pos\n(correct Bad)']
bars = ax.bar(outcomes, [tn, fp, fn, tp],
              color=[C_APPROVE, C_DENY, C_DENY, C_BEST],
              edgecolor='white', width=0.55)
ax.bar_label(bars, padding=3, fontsize=10, fontweight='bold')
ax.set_title(f'Prediction Outcomes (threshold = {THRESHOLD})',
             fontweight='bold', color=C_INK)
ax.set_ylabel('Number of applicants', color=C_INK)
ax.tick_params(axis='x', labelsize=8)
ax.grid(False)

# --- Panel B: DP gap (age) ---------------------------------------------------
ax = axes[1]
bars2 = ax.bar(['Bank decisions', 'Baseline RF'],
               [gap_bank, gap_base],
               color=[C_DENY, C_APPROVE],
               edgecolor='white', width=0.40)
ax.bar_label(bars2, fmt='%.3f', padding=4, fontsize=11, fontweight='bold')
ax.set_title('Demographic Parity Gap — Age\n(lower = more equal treatment)',
             fontweight='bold', color=C_INK)
ax.set_ylabel('Absolute gap in Bad-label rates', color=C_INK)
ax.axhline(0.1, color=C_DENY, linestyle=':', lw=1.2, label='0.1 reference')
ax.set_ylim(0, max(gap_bank, gap_base) * 1.3)
ax.legend(fontsize=9)
ax.grid(False)

plt.suptitle('Stage 2 — Baseline Model: Performance and Fairness (Age)',
             fontsize=12, fontweight='bold', y=1.01, color=C_INK)
plt.tight_layout()
plt.savefig('stage2_baseline_audit.png', bbox_inches='tight')
plt.show()


## 2.7 — Store Results


In [ ]:
baseline_results = {
    'model'      : rf_baseline,
    'y_prob'     : y_prob_baseline,
    'y_pred'     : y_pred_baseline,
    'threshold'  : THRESHOLD,
    'auc'        : auc_test,
    'accuracy'   : acc_baseline,
    'dp_gap_age' : dp_age_base['DP_gap'].iloc[0],
}
bank_results = {
    'auc'        : roc_auc_score(y_gt_test, bank_pred_binary),
    'accuracy'   : accuracy_score(y_gt_test, bank_pred_binary),
    'dp_gap_age' : dp_age_bank['DP_gap'].iloc[0],
}
print("Results stored.")
print(f"  Baseline AUC  : {baseline_results['auc']:.4f}")
print(f"  DP gap (age)  : {baseline_results['dp_gap_age']:.3f}"
      f"  (bank: {bank_results['dp_gap_age']:.3f})")


---
# STAGE 3: Fairness Interventions Across the ML Lifecycle

Based on the EDA finding that age is the primary source of demographic
disparity, all three interventions target the `is_young` attribute.

| Intervention | Method | Core idea |
|---|---|---|
| **Pre-training** | Reweighing | Re-balance training samples by `is_young × label` combination |
| **In-training** | Cost-sensitive weights `{0:1, 1:5}` | Penalise missed Bad labels 5× more during tree construction |
| **Post-training** | Manual group thresholds | Apply a higher flag threshold for young applicants |

Key question: can any intervention bring the model's DP gap (age) **below**
the bank's original level — measured in Stage 2?


## 3.1 — Shared Helpers


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd, numpy as np

def collect_metrics(name, y_true, y_prob, y_pred, s_young):
    """One row of metrics — age DP gap only."""
    auc  = roc_auc_score(y_true, y_prob)
    acc  = accuracy_score(y_true, y_pred)
    dp_y = dp_gap(y_pred, s_young)['DP_gap'].iloc[0]
    return {
        'Model'        : name,
        'ROC-AUC'      : round(auc,  4),
        'Accuracy'     : round(acc,  4),
        'DP gap (age)' : round(dp_y, 4),
    }

young_test = X_test['is_young']
results_log = []

results_log.append(collect_metrics(
    'Bank (original)',
    y_gt_test, bank_pred_binary.astype(float), bank_pred_binary, young_test,
))
results_log.append(collect_metrics(
    'Baseline RF',
    y_gt_test, y_prob_baseline, y_pred_baseline, young_test,
))
print("Helpers ready.")


## 3.2 — Pre-Training: Reweighing on Age

**Idea**: assign a weight to each training example so that the joint
distribution of `(is_young, label)` becomes uniform. This corrects for
the under-representation of young applicants with certain labels before
training begins.


In [ ]:
def compute_reweighing_weights(y: pd.Series, sensitive: pd.Series) -> np.ndarray:
    """Kamiran & Calders (2012) reweighing weights."""
    df_w = pd.DataFrame({'y': y.values, 's': sensitive.values})
    n    = len(df_w)
    p_y  = df_w['y'].value_counts(normalize=True)
    p_s  = df_w['s'].value_counts(normalize=True)
    p_sy = df_w.groupby(['s', 'y']).size() / n
    return df_w.apply(
        lambda row: (p_s[row['s']] * p_y[row['y']]) / p_sy[(row['s'], row['y'])],
        axis=1,
    ).values

young_train = X_train['is_young']
sample_weights_rw = compute_reweighing_weights(y_train, young_train)

print(f"Weight range : {sample_weights_rw.min():.4f} – {sample_weights_rw.max():.4f}")
print(f"Mean weight  : {sample_weights_rw.mean():.4f}  (should be ≈ 1.0)")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_reweigh = RandomForestClassifier(
    n_estimators=250, max_depth=10, min_samples_leaf=7,
    max_features='log2', criterion='entropy',
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_reweigh.fit(X_train, y_train, sample_weight=sample_weights_rw)

y_prob_rw = rf_reweigh.predict_proba(X_test)[:, 1]
y_pred_rw = (y_prob_rw >= THRESHOLD).astype(int)

rw_row = collect_metrics('Pre-Training (Reweighing)',
                          y_gt_test, y_prob_rw, y_pred_rw, young_test)
results_log.append(rw_row)
print(f"Reweighing trained.  AUC={rw_row['ROC-AUC']:.4f}  "
      f"DP(age)={rw_row['DP gap (age)']:.4f}")


## 3.3 — In-Training: Cost-Sensitive Class Weights

**Idea**: instead of letting the model treat all errors equally, we pass the
1:5 cost ratio directly into the Random Forest as `class_weight={0: 1, 1: 5}`.

This tells each decision tree: "a false negative (missed default) is 5 times
as bad as a false positive (unnecessary rejection)." The trees will therefore
bias their splits toward catching more positives — which incidentally also
helps groups that are underrepresented among true defaults.

This is a genuine in-training intervention: the cost enters the learning
algorithm at split time, not via sample pre-processing.


In [ ]:
rf_cost = RandomForestClassifier(
    n_estimators=250, max_depth=10, min_samples_leaf=7,
    max_features='log2', criterion='entropy',
    class_weight={0: 1, 1: 5},
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_cost.fit(X_train, y_train)

y_prob_cost = rf_cost.predict_proba(X_test)[:, 1]
y_pred_cost = (y_prob_cost >= THRESHOLD).astype(int)

cost_row = collect_metrics('In-Training (Cost Weights 1:5)',
                            y_gt_test, y_prob_cost, y_pred_cost, young_test)
results_log.append(cost_row)
print(f"Cost-weighted trained.  AUC={cost_row['ROC-AUC']:.4f}  "
      f"DP(age)={cost_row['DP gap (age)']:.4f}")


## 3.4 — Post-Training: Manual Group Thresholds

**Idea**: after training, we apply a different decision threshold to each
demographic group instead of a single global threshold.

If the model over-flags young applicants (flags them as risky more often
than other applicants with similar actual risk), we can raise the threshold
for that group — requiring higher confidence before flagging — to reduce
the gap.

**How to experiment**: change the values below and re-run this cell.
The comparison table in section 3.5 will update automatically.


In [ ]:
# ── ADJUST THESE VALUES TO EXPERIMENT ────────────────────────────────────────
THRESHOLD_YOUNG = 0.4   # flag threshold for applicants aged <= 25
THRESHOLD_OTHER = 0.30   # flag threshold for all other applicants
# ─────────────────────────────────────────────────────────────────────────────
# Raising THRESHOLD_YOUNG reduces the Bad-label rate for young applicants.
# Try values between 0.25 and 0.55 to explore the fairness tradeoff.

is_young_test = (young_test == 1).values
y_pred_thresh = np.where(
    is_young_test,
    (y_prob_baseline >= THRESHOLD_YOUNG).astype(int),
    (y_prob_baseline >= THRESHOLD_OTHER).astype(int),
)

thresh_row = collect_metrics(
    f'Post-Training (Young={THRESHOLD_YOUNG}, Other={THRESHOLD_OTHER})',
    y_gt_test, y_prob_baseline, y_pred_thresh, young_test,
)
results_log.append(thresh_row)
print(f"Thresholds applied.  AUC={thresh_row['ROC-AUC']:.4f}  "
      f"DP(age)={thresh_row['DP gap (age)']:.4f}")


## 3.5 — Comparison Table

All five variants on three metrics. The bank's DP gap (age) is the
reference — can any intervention fall below it?


In [ ]:
results_df = pd.DataFrame(results_log).drop_duplicates(subset='Model').set_index('Model')

def highlight_best(s):
    higher_better = s.name in ('ROC-AUC', 'Accuracy')
    best = s.max() if higher_better else s.min()
    return [f'font-weight: bold; color: {C_BEST}' if v == best else '' for v in s]

display(results_df.style.apply(highlight_best).format('{:.4f}'))


## 3.6 — Visualisation

Two panels: (A) ROC-AUC across models, (B) DP gap (age) across models.
The red dashed line in panel B marks the bank's own DP gap — the fairness reference.


In [ ]:
model_colors = [C_INK, C_APPROVE, C_BEST, '#8B00FA', C_DENY]
models = results_df.index.tolist()
x      = np.arange(len(models))
bar_w  = 0.55

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=C_BG)

# Panel A: ROC-AUC
ax = axes[0]
bars = ax.bar(x, results_df['ROC-AUC'], color=model_colors,
              edgecolor='white', width=bar_w)
ax.axhline(results_df.loc['Baseline RF', 'ROC-AUC'],
           color='grey', linestyle='--', lw=1.2,
           label=f"Baseline ({results_df.loc['Baseline RF','ROC-AUC']:.3f})")
ax.set_xticks(x)
ax.set_xticklabels([m.replace('(', '\n(') for m in models],
                   rotation=15, ha='right', fontsize=8)
ax.set_title('ROC-AUC', fontweight='bold', color=C_INK)
ax.set_ylabel('AUC score', color=C_INK)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
ax.legend(fontsize=8)
ax.grid(False)

# Panel B: DP gap (age)
ax = axes[1]
bars = ax.bar(x, results_df['DP gap (age)'], color=model_colors,
              edgecolor='white', width=bar_w)
bank_dp_ref = results_df.loc['Bank (original)', 'DP gap (age)']
ax.axhline(bank_dp_ref, color=C_DENY, linestyle='--', lw=1.5,
           label=f'Bank level ({bank_dp_ref:.3f})')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('(', '\n(') for m in models],
                   rotation=15, ha='right', fontsize=8)
ax.set_title('Demographic Parity Gap — Age\n(lower = more equal treatment)',
             fontweight='bold', color=C_INK)
ax.set_ylabel('Absolute gap in Bad-label rates', color=C_INK)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
ax.legend(fontsize=8)
ax.grid(False)

plt.suptitle('Stage 3 — Intervention Comparison (Focus: Age)',
             fontsize=12, fontweight='bold', y=1.01, color=C_INK)
plt.tight_layout()
plt.savefig('stage3_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
stage3_results = results_df.copy()
print("stage3_results saved — ready for Stage 4.")


---
# STAGE 4: Feature Importance

Which features does the baseline model rely on most when replicating the
bank's decisions?

If demographic features (age, sex) rank highly, it means the model has learned
to use group membership as a signal — which is the mechanism behind the DP gaps
measured in Stages 2 and 3.


In [ ]:
# Feature importance from the baseline Random Forest.
# One-hot encoding creates multiple columns per original attribute (e.g. A11/A12/A13
# all map to "Checking Account Status"). We sum their MDI shares back to the
# original attribute level for a cleaner chart.
# C_DENY (red) = demographic features. C_APPROVE (blue-violet) = financial features.

raw_imp = pd.Series(rf_baseline.feature_importances_, index=X_train.columns)

# Map each encoded column back to its original attribute name
def original_attr(col):
    """Return the original attribute key for an encoded column."""
    for key in ATTR_LABEL:
        if col == key or col.startswith(key + '_'):
            return key
    return col  # already a plain name (e.g. is_young, sex_male)

grouped_imp = (
    raw_imp
    .groupby(raw_imp.index.map(original_attr))
    .sum()
    .sort_values(ascending=False)
)

# Keep only features with at least 1% combined MDI share
importances_top = grouped_imp[grouped_imp > 0.01].head(15)

demo_feats = {'Attribute13', 'is_young', 'sex_male', 'sex_female',
              'Attribute9',  # marital/sex combined — already decomposed
              'marital_status_single', 'marital_status_married/widowed',
              'marital_status_divorced/separated',
              'marital_status_divorced/separated/married'}

bar_colors = [C_DENY if f in demo_feats else C_APPROVE
              for f in importances_top.index]

readable_labels = [readable(f) for f in importances_top.index[::-1]]

fig, ax = plt.subplots(figsize=(11, 6), facecolor=C_BG)
bars = ax.barh(readable_labels, importances_top.values[::-1],
               color=bar_colors[::-1], edgecolor='white')
ax.set_xlabel('Aggregated MDI share (summed across dummy columns)',
              fontsize=10, color=C_INK)
ax.set_title(
    'Feature Importance — Baseline Random Forest\n'
    '(red = demographic features,  blue-violet = financial features)',
    fontweight='bold', color=C_INK,
)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
ax.set_xlim(0, importances_top.values.max() * 1.20)
ax.grid(False)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=C_APPROVE, label='Financial / behavioural'),
    Patch(facecolor=C_DENY,    label='Demographic (age / sex)'),
], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('stage4_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nTop features by aggregated MDI share (>1%):")
for feat, val in importances_top.items():
    label = readable(feat)
    tag   = ' [demographic]' if feat in demo_feats else ''
    print(f"  {label:<38} {val:.4f}{tag}")


**What this tells us:** if demographic features (age, sex) appear high on
this chart, the model is partially making decisions based on who the applicant
*is* rather than how they *behave financially*. This is the mechanism behind
the DP gaps measured in Stages 2 and 3.


---
# STAGE 5: Synthesis and Conclusions

We consolidate all results and answer the central question of this project:

> Does a machine learning model trained on historical bank decisions inherit
> and amplify the demographic bias present in those decisions — and can
> fairness interventions reduce this effect?


## 5.1 — Approval Rate by Age Group

The chart shows the share of applicants **not** flagged as Bad across
all five model variants, split by age group.

- Gap annotation in **red** = difference > 5 pp (fairness concern)
- Gap annotation in **green** = within 5 pp (acceptable range)


In [ ]:
all_preds = {
    'Bank (original)'          : bank_pred_binary,
    'Baseline RF'              : y_pred_baseline,
    'Pre-Training (Reweighing)': y_pred_rw,
    'In-Training (Cost 1:5)'   : y_pred_cost,
    f'Post-Training\n(Young={THRESHOLD_YOUNG}, Other={THRESHOLD_OTHER})': y_pred_thresh,
}
model_colors_5 = [C_INK, C_APPROVE, C_BEST, '#8B00FA', C_DENY]

sr_rows = []
for name, preds in all_preds.items():
    approved = (pd.Series(preds, index=y_gt_test.index) == 0).astype(int)
    sr_rows.append({
        'Model'    : name,
        'SR young' : approved[young_test == 1].mean(),
        'SR other' : approved[young_test == 0].mean(),
    })
sr_df = pd.DataFrame(sr_rows)

x     = np.arange(len(sr_df))
bar_w = 0.32

fig, ax = plt.subplots(figsize=(12, 5), facecolor=C_BG)
ax.bar(x - bar_w/2, sr_df['SR young'], width=bar_w,
       color=model_colors_5, edgecolor='white', alpha=0.95)
ax.bar(x + bar_w/2, sr_df['SR other'], width=bar_w,
       color=model_colors_5, edgecolor='white', alpha=0.50)

for i in range(len(sr_df)):
    gap = sr_df['SR other'].iloc[i] - sr_df['SR young'].iloc[i]
    top = max(sr_df['SR young'].iloc[i], sr_df['SR other'].iloc[i]) + 0.025
    clr = C_DENY if abs(gap) > 0.05 else C_BEST
    ax.annotate(f'D={gap:+.3f}', xy=(x[i], top),
                ha='center', va='bottom', fontsize=9,
                color=clr, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(sr_df['Model'].tolist(), rotation=12, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.15)
ax.set_ylabel('Share not flagged as Bad', fontsize=10, color=C_INK)
ax.set_title('Approval Rate by Age Group — Before and After Interventions',
             fontweight='bold', color=C_INK)
ax.grid(False)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='grey', alpha=0.95, label='Young (Age <= 25)'),
    Patch(facecolor='grey', alpha=0.50, label='Other (Age > 25)'),
], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('stage5_selection_rates.png', bbox_inches='tight', dpi=150)
plt.show()


## 5.2 — Final Consolidated Summary Table


In [ ]:
summary_df = stage3_results.copy()

def style_cell(s):
    higher_better = s.name in ('ROC-AUC', 'Accuracy')
    col_min, col_max = s.min(), s.max()
    span  = col_max - col_min if col_max != col_min else 1e-9
    cmap  = plt.cm.RdYlGn
    styles = []
    for v in s:
        norm  = (v - col_min) / span if higher_better else 1 - (v - col_min) / span
        rgba  = cmap(norm)
        hex_c = '#{:02x}{:02x}{:02x}'.format(
            int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))
        lum   = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
        styles.append(
            f'background-color: {hex_c}; color: {"black" if lum > 0.45 else "white"}')
    return styles

print("=" * 55)
print("  FINAL EVALUATION — ALL MODELS")
print("=" * 55)
display(summary_df.style.apply(style_cell).format('{:.4f}'))


## 5.3 — Written Conclusions


In [ ]:
bank_dp  = stage3_results.loc['Bank (original)', 'DP gap (age)']
base_dp  = stage3_results.loc['Baseline RF',     'DP gap (age)']

excl_bank = [m for m in stage3_results.index if m != 'Bank (original)']
best_m    = stage3_results.loc[excl_bank, 'DP gap (age)'].idxmin()
best_dp   = stage3_results.loc[best_m, 'DP gap (age)']
best_auc  = stage3_results.loc[best_m, 'ROC-AUC']

print("STAGE 5 — SYNTHESIS AND CONCLUSIONS")
print("=" * 55)

print("\n1. DOES THE MODEL INHERIT BANK BIAS?\n")
print(f"   Bank DP gap (age)     : {bank_dp:.4f}")
print(f"   Baseline model DP gap : {base_dp:.4f}  "
      f"(+{base_dp - bank_dp:.4f} vs bank — amplified)")
print("\n   A model trained on historical bank decisions reproduces")
print("   and amplifies the age-based disparity in the training data.")

print("\n2. DO INTERVENTIONS REDUCE BIAS?\n")
for m in excl_bank:
    dp  = stage3_results.loc[m, 'DP gap (age)']
    auc = stage3_results.loc[m, 'ROC-AUC']
    vs  = 'below bank level' if dp < bank_dp else 'above bank level'
    tag = '  <-- best' if m == best_m else ''
    print(f"   {m:<48} DP={dp:.4f}  ({vs}){tag}")

print("\n3. INTERPRETATION OF EACH INTERVENTION\n")
print("   Pre-Training (Reweighing):")
print("     Re-balances training data by is_young x label combination.")
print("     Directly targets the age gap before the model sees the data.")
print()
print("   In-Training (Cost Weights 1:5):")
print("     Penalises missed Bad labels more heavily during tree construction.")
print("     Affects overall recall/precision — not specifically age distribution.")
print()
print("   Post-Training (Group Thresholds):")
print("     Raises the flag threshold for young applicants directly.")
print("     Most direct tool for closing the age gap. Experiment with")
print("     THRESHOLD_YOUNG in Stage 3.4 to explore the tradeoff.")

print(f"\n4. KEY TAKEAWAY\n")
print(f"   Bank DP gap (age) = {bank_dp:.3f}")
print(f"   Baseline model    = {base_dp:.3f}  (inherited + amplified)")
print(f"   Best intervention = {best_dp:.3f}  ({best_m})")
status = 'below' if best_dp < bank_dp else 'still above'
print(f"   Result: {status} the bank's own level of {bank_dp:.3f}.")
print("\nStage 5 complete.")
